In [ ]:
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-xyrkal0l
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-xyrkal0l
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 28f872a2f99a1b201bcd0db14fdbc5a496b9bfd7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nvcc4jupyter: filename=nvcc4jupyter-1.2.1-py3-none-any.whl size=10742 sha256=fdb30b442a03a2b978d94f242f4f3839c3b517111d4c5bcc31496f40b5aacbba
  Stored in directory: /tmp/pip-ephem-wheel-cache-bb7e1vwz/wheels/7d/b9/66/459b9938664e6a93d1a85323ec52f7e51cd7265d253410a7d8
Successfully built nvcc4jupyter


In [ ]:
!nvidia-smi

Sat Oct 18 11:05:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!apt-get update
!apt-get install -y cuda-toolkit-12-2


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,812 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,374 kB]
Fetched 12.6 MB in 2s (6,2

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [ ]:
%%writefile matrix_mul.cu
#define TILE_SIZE 4   // You can tune this (8, 16, or 32 depending on GPU)

// Tiled Matrix Multiplication Kernel
__global__ void matrix_mul(int *a, int *b, int *c, int n) {
    // Shared memory tiles for sub-blocks of A and B
    __shared__ int tileA[TILE_SIZE][TILE_SIZE];
    __shared__ int tileB[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;

    int temp = 0;

    // Loop over all tiles required to compute C[row][col]
    for (int m = 0; m < (n + TILE_SIZE - 1) / TILE_SIZE; m++) {
        // Load elements of A into shared memory
        if (row < n && (m * TILE_SIZE + threadIdx.x) < n)
            tileA[threadIdx.y][threadIdx.x] = a[row * n + (m * TILE_SIZE + threadIdx.x)];
        else
            tileA[threadIdx.y][threadIdx.x] = 0;

        // Load elements of B into shared memory
        if (col < n && (m * TILE_SIZE + threadIdx.y) < n)
            tileB[threadIdx.y][threadIdx.x] = b[(m * TILE_SIZE + threadIdx.y) * n + col];
        else
            tileB[threadIdx.y][threadIdx.x] = 0;

        __syncthreads();  // Synchronize to ensure all threads have loaded their tile

        // Multiply current tile
        for (int k = 0; k < TILE_SIZE; k++) {
            temp += tileA[threadIdx.y][k] * tileB[k][threadIdx.x];
        }

        __syncthreads();  // Wait before loading new tiles
    }

    // Store result
    if (row < n && col < n)
        c[row * n + col] = temp;
}


Writing matrix_mul.cu


In [ ]:
%%writefile mat-mul.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include "matrix_mul.cu"
#define N 5

int main() {
    int n = N;
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = n * n * sizeof(int);

    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++) {
            a[i * n + j] = i + j;
            b[i * n + j] = i * j;
        }

    // print original matrixes
    printf("First Matrix is as below\n");
    for (int i = 0; i < n; i++){
        for (int j = 0; j < n; j++)
            printf("%d ", a[i * n + j]);
        printf("\n");
        }

    printf("Second Matrix is as below\n");
    for (int i = 0; i < n; i++){
        for (int j = 0; j < n; j++)
            printf("%d ", b[i * n + j]);
        printf("\n");
        }

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Choose TILE_SIZE × TILE_SIZE threads per block
    dim3 blockSize(TILE_SIZE, TILE_SIZE);

    // Grid size to cover the full matrix
    dim3 gridSize((n + TILE_SIZE - 1) / TILE_SIZE,
                  (n + TILE_SIZE - 1) / TILE_SIZE);

    matrix_mul<<<gridSize, blockSize>>>(d_a, d_b, d_c, n);


    cudaError_t err = cudaGetLastError();
    cudaDeviceSynchronize();
    if (err != cudaSuccess)
         printf("CUDA error: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);
    printf("Product Matrix is as below\n");
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++)
            printf("%d ", c[i * n + j]);
        printf("\n");
    }

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(a);
    free(b);
    free(c);

    return 0;
}

Writing mat-mul.cu


In [ ]:
!nvcc mat-mul.cu -o mat_mul -arch=sm_75

In [ ]:
!./mat_mul

First Matrix is as below
0 1 2 3 4 
1 2 3 4 5 
2 3 4 5 6 
3 4 5 6 7 
4 5 6 7 8 
Second Matrix is as below
0 0 0 0 0 
0 1 2 3 4 
0 2 4 6 8 
0 3 6 9 12 
0 4 8 12 16 
Product Matrix is as below
0 30 60 90 120 
0 40 80 120 160 
0 50 100 150 200 
0 60 120 180 240 
0 70 140 210 280 


In [ ]:
# 1️⃣  Go to a temp directory
%cd /content

# 2️⃣  Download Nsight Compute CLI .deb for CUDA 12.5 (from NVIDIA developer repo)
!wget https://developer.download.nvidia.com/compute/cuda/12.5.39/local_installers/cuda-nsight-compute-12-5_12.5.39-1_amd64.deb

# 3️⃣  Install it
!sudo dpkg -i cuda-nsight-compute-12-5_12.5.39-1_amd64.deb

# 4️⃣  Verify installation
!nv-nsight-cu-cli --version


/content
--2025-10-18 15:45:27--  https://developer.download.nvidia.com/compute/cuda/12.5.39/local_installers/cuda-nsight-compute-12-5_12.5.39-1_amd64.deb
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.46.17.66, 23.46.17.43
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.46.17.66|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-10-18 15:45:27 ERROR 404: Not Found.

dpkg: error: cannot access archive 'cuda-nsight-compute-12-5_12.5.39-1_amd64.deb': No such file or directory
/bin/bash: line 1: nv-nsight-cu-cli: command not found


In [ ]:
%cd /content

/content


In [ ]:
!wget https://developer.download.nvidia.com/compute/cuda/12.5.39/local_installers/cuda-nsight-compute-12-5_12.5.39-1_amd64.deb


--2025-10-18 17:03:15--  https://developer.download.nvidia.com/compute/cuda/12.5.39/local_installers/cuda-nsight-compute-12-5_12.5.39-1_amd64.deb
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.15.241.65, 23.15.241.19
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.15.241.65|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-10-18 17:03:16 ERROR 404: Not Found.



In [ ]:
!apt-get install -y cuda-toolkit-13-0


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cuda-toolkit-13-0 is already the newest version (13.0.2-1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


In [ ]:
!which nv-nsight-cu-cli